In [ ]:
import pandas as pd
df = pd.read_json('Varad_data.jsonl', lines=True)
print(df.head)

In [ ]:
import torch
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

special_tokens = {
    "<|user|>": 200100,
    "<|assistant|>": 200101,
    "<|mood|>" : 200103 ,
    "<|end|>": 200102
}
allowed = set(special_tokens.keys())

user_messege = []
asistant_reply = []
mood = []

for i in range(len(df['messages'])) :
    user_messege.append(df['messages'][i][0]['content'] )
    mood.append(df['messages'][i][1].get('mood', 'happy').lower())
    asistant_reply.append(df['messages'][i][1]['content'])

question_tokens  = []
target_tokens = []

for i in range(len(df['messages'])) :
    user_encods = enc.encode( "<|user|>"  + user_messege[i] ,  allowed_special= allowed)
    asis_encods = enc.encode( "<|mood|>" + mood[i] +  "<|assistant|>" +  asistant_reply[i] + "<|end|>"  , allowed_special= allowed )
    tokens = user_encods + asis_encods
    question = torch.tensor(tokens[:-1])
    question_tokens.append(question)
    target    = torch.tensor(tokens[1:])
    target_tokens.append(target)



In [ ]:
g = torch.Generator()
g.manual_seed(42)

In [ ]:
max_num = max(set(tokens))
print(max_num)

In [ ]:
vocab_size = len(set(tokens))
print(vocab_size)

In [ ]:
compl_token = []
for i in range(len(question_tokens)):
    compl_token += question_tokens[i].tolist()
    compl_token += target_tokens[i].tolist()

vocab = sorted(set(compl_token))
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for i,s in enumerate(vocab)}
vocab_size = len(vocab)
q_tokens = []
t_tokens = []
for i in range(len(question_tokens)):
    q_tensor = torch.tensor([stoi[tok.item()] for tok in question_tokens[i]])
    t_tensor = torch.tensor([stoi[tok.item()] for tok in target_tokens[i]])
    q_tokens.append(q_tensor)
    t_tokens.append(t_tensor)

In [ ]:
q_tokens = torch.cat(q_tokens)
t_tokens = torch.cat(t_tokens)

In [ ]:
block_size = 64
xs = []
ys = []

for i in range(len(q_tokens)):
    context = q_tokens[max(0, i-block_size+1) : i+1]
    context = torch.nn.functional.pad(context, (block_size - len(context), 0))
    xs.append(context)
    ys.append(t_tokens[i])

xs = torch.stack(xs)
ys = torch.stack(ys)

In [ ]:
dim = 300

Encods  = torch.randn((vocab_size, dim), requires_grad=True)
W1 = torch.randn((64*dim, dim), requires_grad=True)
W2 = torch.randn((dim, vocab_size), requires_grad=True)
enc = Encods[xs]
update_enc = enc.view(enc.shape[0] , -1 )
hidden = torch.tanh(update_enc @ W1 )
logits = hidden @ W2

In [ ]:
b_gain = torch.ones((1, 300), requires_grad=True)
b_bias = torch.zeros((1, 300), requires_grad=True)
parameters  = { Encods , W1 , W2  , b_gain , b_bias}

In [ ]:
logits.shape

In [ ]:
loss = torch.nn.functional.cross_entropy(logits, ys)
print(loss)

In [ ]:
from google.colab import files
uploaded = files.upload()

checkpoint = torch.load('Sakhi.pt')
Encods      = checkpoint['Encods']
W1     = checkpoint['W1']
W2     = checkpoint['W2']
b_gain = checkpoint['b_gain']
b_bias = checkpoint['b_bias']
stoi   = checkpoint['stoi']
itos   = checkpoint['itos']

vocab_size = len(stoi)
parameters  = { Encods , W1 , W2  , b_gain , b_bias}

In [ ]:
batch_size = 32
for i in range(10000):
    ix = torch.randint(0, xs.shape[0], (batch_size,))
    enc  = Encods[xs[ix]]
    update_enc = enc.view(enc.shape[0], -1)
    flow_layer = update_enc @ W1
    flow_layer = b_gain * (flow_layer - flow_layer.mean(0)) / flow_layer.std(0) + b_bias
    hidden     = torch.tanh(flow_layer)
    logits     = hidden @ W2
    loss       = torch.nn.functional.cross_entropy(logits, ys[ix])

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad


print(loss)

In [ ]:
torch.save({'Encods' : Encods,'W1'     : W1,'W2'     : W2,'b_gain' : b_gain,    'b_bias' : b_bias,'stoi'   : stoi,'itos'   : itos,}, 'Amika.pt')
from google.colab import files
files.download('Sakhi.pt')